# Alice EEG with deepSTRF — modality generalisation tutorial

<a href="https://colab.research.google.com/github/urancon/deepSTRF/blob/develop/examples/alice_eeg_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook applies deepSTRF — originally built for single-unit recordings of the auditory pathway — to a **scalp EEG** dataset. We use the *Alice* corpus released by Bhattasali et al. (2020) and preprocessed by Brodbeck et al. (2023, [eLife](https://doi.org/10.7554/eLife.85012)) for their Eelbrain toolkit paper. The same library that ships LN, AdapTrans, and DNN cores for spiking auditory cortex turns out to plug right in.

## What's in the dataset

33 human participants listened to the first chapter of *Alice in Wonderland* (12.4 minutes, split into 12 audio segments) while 61-channel EEG was recorded. Brodbeck et al. demonstrate that linear temporal response functions (TRFs, the EEG analogue of an STRF) explain up to ~20 % of the variability in the EEG response to gammatone spectrograms plus acoustic-onset predictors.

## What we'll do here

1. Load the Alice EEG dataset via the deepSTRF paradigm.
2. Visualise one audio segment alongside the EEG response (the eelbrain paper's Fig 8 analog).
3. Train three encoding models with increasing flexibility:
   - **Linear** baseline on the 8-band gammatone spectrogram (TRF analog).
   - **AdapTrans + Linear** — the deepSTRF reframing of Brodbeck's onset-spectrogram condition: a learnable peripheral adaptation front-end in place of the hand-engineered onset detector.
   - **AdapTrans + ConvNet2D** — a full DNN core for comparison.
4. Report per-channel fraction-of-variability-explained (fve), the same metric as Brodbeck Fig 4.

By default we fit on a single subject (≈1-minute total runtime on a CPU/Colab CPU). An optional final cell scales up to the full 33-subject population.

In [ ]:
# Colab-friendly install. Skipped on local installations.
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q 'deepSTRF[eeg] @ git+https://github.com/urancon/deepSTRF.git@develop'

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

from deepSTRF.datasets.audio import Alice_EEG_Dataset
from deepSTRF.models.audio import Linear, ConvNet2D
from deepSTRF.models.prefiltering import make_prefiltering
from deepSTRF.utils.data import neural_collate
from deepSTRF.metrics import mse_loss, corrcoef, fve

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

## 1. Load the dataset

We use a single subject (`S20`) by default — same subject Brodbeck shows in their figures. `download=True` pulls the ~2.5 GiB Brodbeck restructure from UMd DRUM into the platform cache directory; replace with `path=...` if you've already downloaded the data.

In [ ]:
SUBJECT = "S20"  # change to e.g. "S01" if a different subject is preferred

ds = Alice_EEG_Dataset(
    download=True,           # set False + path=... if data is local
    subjects=[SUBJECT],
    dt_ms=10.0,              # 100 Hz, matches Brodbeck 2023
    n_frequency_bands=8,     # ERB-band gammatone approximation
)
print(f"S={len(ds.stims)} segments, N={ds.N_neurons} channels, F={ds.F} bands, dt={ds.dt} ms")
print(f"first segment: spectrogram {ds.stims[0].shape}, EEG {ds.responses[0][0].shape}")
total_min = sum(m['duration_s'] for m in ds.stim_meta) / 60
print(f"total audio: {total_min:.2f} min")

## 2. Visualise the stimulus / response (Brodbeck Fig 8 analog)

For a 6-second slice of the first segment: the ERB-band log spectrogram, its summed envelope, and a representative EEG channel.

In [ ]:
spec = ds.stims[0][0]                       # (F, T)
ch_idx = next(
    i for i, m in enumerate(ds.neuron_metadata)
    if m['channel_id'] == '40' and not ds.responses[0][i].isnan().all()
)
eeg = ds.responses[0][ch_idx][0]            # (T,) — first repeat
envelope = spec.exp().sum(dim=0)            # broadband acoustic energy
T = min(600, spec.shape[-1])                # 6 seconds at 100 Hz
t_s = np.arange(T) * ds.dt / 1000

fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
axes[0].imshow(spec[:, :T], aspect='auto', origin='lower',
               extent=[0, t_s[-1], 0, ds.F], cmap='magma')
axes[0].set_ylabel('ERB band'); axes[0].set_title(f'Subject {SUBJECT}, segment 1')
axes[1].plot(t_s, envelope[:T]); axes[1].set_ylabel('Envelope')
axes[2].plot(t_s, eeg[:T]); axes[2].set_ylabel(f'EEG ch {ds.neuron_metadata[ch_idx]["channel_id"]}')
axes[2].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## 3. Standardise responses and split train / test

EEG amplitudes are O(1 μV) — z-scoring per channel (using *training* statistics only) keeps the optimisation well-conditioned. We hold out segment 12 as the test set.

In [ ]:
TRAIN_IDX = list(range(11))
TEST_IDX = [11]

# Per-channel mean and std from training segments, ignoring NaN sentinels.
mean = torch.zeros(ds.N_neurons)
std = torch.ones(ds.N_neurons)
for n in range(ds.N_neurons):
    chunks = [ds.responses[s][n].flatten() for s in TRAIN_IDX
              if ds.responses[s][n].shape[-1] > 1]
    if not chunks:
        continue
    cat = torch.cat(chunks)
    if cat.isnan().all() or cat.numel() < 2:
        continue
    mean[n] = torch.nanmean(cat)
    std[n] = max(float(np.nanstd(cat.numpy())), 1e-9)

# Apply in place — preserves the (1, 1) NaN sentinel for fully-bad channels.
for s in range(len(ds.responses)):
    for n in range(ds.N_neurons):
        r = ds.responses[s][n]
        if r.shape == (1, 1):
            continue
        ds.responses[s][n] = (r - mean[n]) / std[n]

train_loader = DataLoader(Subset(ds, TRAIN_IDX), batch_size=2,
                          shuffle=True, collate_fn=neural_collate)
test_loader  = DataLoader(Subset(ds, TEST_IDX),  batch_size=1,
                          collate_fn=neural_collate)

## 4. Train three models

Three encoding models with progressively more flexibility:
- **Linear** baseline — a single STRF over the gammatone spectrogram, the deepSTRF analogue of a multivariate TRF.
- **AdapTrans + Linear** — a learnable peripheral-adaptation front-end (Rançon et al. 2025) before the same STRF readout. This is the deepSTRF reframing of Brodbeck Fig 4D's onset-spectrogram condition.
- **AdapTrans + ConvNet2D** — a small 2D CNN core for the nonlinear comparison.

All three use signed-target identity output and mean-squared-error loss (signed EEG, not spike counts).

In [ ]:
F, N = ds.F, ds.N_neurons
T_WIN = 50         # 500 ms STRF — matches Brodbeck's max lag
N_EPOCHS = 80
LR = 3e-3

def fit_and_evaluate(model, n_epochs=N_EPOCHS, lr=LR):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    train_losses = []
    model.train()
    for ep in range(n_epochs):
        ep_losses = []
        for stims, responses, _, _ in train_loader:
            stims, responses = stims.to(device), responses.to(device)
            pred = model(stims)
            loss = mse_loss(pred, responses)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_losses.append(loss.item())
        train_losses.append(np.mean(ep_losses))
    # held-out test set
    model.eval()
    with torch.no_grad():
        for stims, responses, _, _ in test_loader:
            stims, responses = stims.to(device), responses.to(device)
            pred = model(stims)
            gt_psth = responses.nanmean(dim=2, keepdim=True)
            cc = corrcoef(pred, gt_psth, reduction='none').cpu()
            var = fve(pred, gt_psth, reduction='none').cpu()
    return {'train_loss': train_losses, 'cc': cc, 'fve': var}

results = {}
for name, build in [
    ('Linear',
        lambda: Linear(F, T_WIN, N, output_activation=torch.nn.Identity())),
    ('AdapTrans + Linear',
        lambda: Linear(F, T_WIN, N, output_activation=torch.nn.Identity(),
                       prefiltering=make_prefiltering('adaptrans', F, dt=ds.dt))),
    ('AdapTrans + ConvNet2D',
        lambda: ConvNet2D(F, kernel_size=(3, 9), out_neurons=N,
                          output_activation=torch.nn.Identity(),
                          prefiltering=make_prefiltering('adaptrans', F, dt=ds.dt))),
]:
    print(f'fitting {name}…', end=' ', flush=True)
    results[name] = fit_and_evaluate(build())
    cc_mean = results[name]['cc'].nanmean().item()
    print(f'test CC = {cc_mean:+.3f}')

## 5. Per-channel fve — compare against Brodbeck Fig 4

Brodbeck reports, averaged across 33 subjects: ~14 % fve for envelope alone, ~17 % for envelope + onsets, ~20 % for full gammatone + onset spectrogram. Our single-subject single-segment numbers are noisier, but the model ordering replicates the predictor-richness ordering.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(results))
means = [r['fve'].nanmean().item() for r in results.values()]
sems = [r['fve'][~r['fve'].isnan()].std().item() / np.sqrt((~r['fve'].isnan()).sum().item())
        for r in results.values()]
ax.bar(x, means, yerr=sems, capsize=4, color=['#4c72b0', '#dd8452', '#55a868'])
ax.axhline(0.14, ls='--', lw=1, color='gray', label='Brodbeck: envelope (~14%)')
ax.axhline(0.20, ls='--', lw=1, color='black', label='Brodbeck: spectrogram + onsets (~20%)')
ax.set_xticks(x); ax.set_xticklabels(list(results.keys()), rotation=15, ha='right')
ax.set_ylabel('fve (mean ± SEM across 61 channels)')
ax.set_title(f'Subject {SUBJECT} — single-segment hold-out fve')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

## 6. Caveats and next steps

This notebook is a **library-functionality demonstration**, not a paper-grade reproduction. To match Brodbeck's headline numbers, several things should be tightened:

1. **Multi-subject training.** Brodbeck reports group means across 33 subjects. Re-instantiate with `subjects=None` and either train one model per subject (parallelisable) or fit jointly with a per-subject readout. The latter is the deepSTRF idiom — set `treat_subjects_as='neurons'` (default) and the `N` axis becomes `61 × n_subjects` automatically.
2. **k-fold cross-validation.** Brodbeck rotates the held-out segment 12 ways. The single-test-segment number above is high-variance.
3. **Hyper-parameter search.** STRF window length, regularisation, learning rate, AdapTrans initialisation all want tuning per dataset.
4. **Predictor library.** Word-onset impulses and n-gram surprisal (Brodbeck Fig 5) are not yet wired into the dataset class — see the `stimuli/AliceChapterOne-EEG.csv` table for the raw predictors.

## Optional: subjects-as-repeats mode

deepSTRF exposes an alternate view of this dataset: treat each subject as a repeat of a shared canonical EEG response. This is the inter-subject-consistency (ISC) framing of the same data and enables `normalized_corrcoef(method='schoppe')` as a model quality metric against the inter-subject reliability ceiling.

```python
ds_isc = Alice_EEG_Dataset(download=True, treat_subjects_as='repeats')
# N = 61 channels, R = 33 subjects per (channel, segment)
```

See [`docs/_source/md/README_Alice_EEG.md`](../docs/_source/md/README_Alice_EEG.md) for the interpretive caveat — this is *inter-subject* reliability, not single-trial reliability, and the Schoppe correction's noise model only approximately holds.